In [1]:
# !git clone https://github.com/Berkeley-CS182/cs182fa25_public.git

In [2]:
!pwd

/home/nico/cs182/hw06/code


In [3]:
%cd /home/nico/cs182/hw06/code

/home/nico/cs182/hw06/code


In [4]:
!pip install wandb tqdm

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import wandb
from architectures import BasicConvNet, ResNet18, MLP
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

%load_ext autoreload
%autoreload 2

# Exploring Tensorboard
Tensorboard is a local tool for visualizing images, metrics, histograms, and more. It is designed for tensorflow, but can be integrated with torch. Let's explore tensorboard usage with an example:

```python
from torch.utils.tensorboard import SummaryWriter

# To start a run, call the following
writer = SummaryWriter(comment=f'Name_of_Run')

# When you want to log a value, use the writer. When adding a scalar, the format is as follows:
# add_scalar(tag, scalar_value, global_step=None, walltime=None, new_style=False, double_precision=False)
writer.add_scalar('Training Loss', loss.item(), step)

# Finally, when you are done logging values, close the writer
writer.close()
```
There are many other functionalities and methods that you are free to explore, but will not be mentioned in this notebook.

## Your Task
We will be once again building classifiers for the CIFAR-10. There are various architectures set up for you to use in the architectures.py file. Using tensorboard, please search through 5 different hyperparameter configurations. Examples of choices include: learning rate, batch size, architecture, optimization algorithm, etc. Please submit the generated plots on your pdf and answer question A.

In [6]:
epochs = 2
cuda = torch.cuda.is_available()
device = torch.device("cuda" if cuda else "cpu")
print(device)

cuda


In [7]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                    download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

In [8]:
hyperparameters = torch.linspace(1e-1, 1e-5, 5)
print(hyperparameters)

tensor([1.0000e-01, 7.5002e-02, 5.0005e-02, 2.5008e-02, 1.0000e-05])


In [14]:
from torch.utils.tensorboard import SummaryWriter
from architectures import BasicConvNet, MLP, ResNet18
SummaryWriter.add_scalar


def train_with_lr(learning_rate, batch_size=64, model_arch='cnn'):
    
    
    # Select model architecture
    if model_arch == 'mlp':
        model = MLP()
    elif model_arch == 'cnn':
        model = BasicConvNet()
    elif model_arch == 'resnet':
        model = ResNet18()
    model.to(device)

    # Initialize training infrastructure
    loss_fn = torch.nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        params=model.parameters(),
        lr=learning_rate
    )

    # Initialize trainloader
    trainloader = torch.utils.data.DataLoader(
        dataset=trainset,
        shuffle=True,
        batch_size=batch_size
    )

    # create writer for TensorBoard
    run_name = f'{model_arch=}.{learning_rate=}.{batch_size=}'
    print(run_name)
    # writer = SummaryWriter(comment=run_name)
    for batch_idx, data in tqdm(enumerate(trainloader), unit=' batch'):
        # extract input + label tensors for this batch
        inputs = data[0].to(device)
        labels = data[1].to(device)

        # reset gradients for this batch
        optimizer.zero_grad()

        # make predictions for this batch
        outputs = model(inputs)

        # compute loss between ground truth and predicted labels
        loss = loss_fn(outputs, labels)

        # do backprop
        loss.backward()

        # adjust learning weights
        optimizer.step()
    
        # When you want to log a value, use the writer. When adding a scalar, the format is as follows:
        # add_scalar(tag, scalar_value, global_step=None, walltime=None, new_style=False, double_precision=False)
        # if batch_idx > 20:
            # writer.add_scalar('Training Loss', loss.item(), global_step=batch_idx)
    
    # Finally, when you are done logging values, close the writer
    # writer.close()
    # raise NotImplementedError

def run():
    for lr in hyperparameters:
        train_with_lr(lr, batch_size=1, model_arch='resnet')
    
# train_with_lr(hyperparameters[0], batch_size=128)
run()

model_arch='resnet'.learning_rate=tensor(0.1000).batch_size=1024


49 batch [00:07,  6.43 batch/s]


model_arch='resnet'.learning_rate=tensor(0.0750).batch_size=1024


49 batch [00:07,  6.51 batch/s]


model_arch='resnet'.learning_rate=tensor(0.0500).batch_size=1024


43 batch [00:06,  6.38 batch/s]


KeyboardInterrupt: 

In [19]:
!nvidia-smi

Sat Oct 25 19:47:28 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070        Off |   00000000:17:00.0 Off |                  N/A |
|  0%   31C    P1             60W /  250W |    2256MiB /  12227MiB |      4%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----